# Полный проход детекции: вход сети 1280

По пробному прогону вход 1280 находит больше людей в толпе и вдали и почти ничего не теряет. Вход 1920 теряет близких людей, поэтому не берём.

1. Меню **Runtime → Change runtime type → A100 GPU** (если её нет, то L4). Если открыт другой блокнот с работающей средой, сначала остановите её: **Runtime → Disconnect and delete runtime** в том блокноте.
2. Файл `mct_colab.zip` уже лежит на Диске в папке `mct`, заново загружать не нужно.
3. **Runtime → Run all**, разрешите доступ к Диску.
4. Если сессия оборвётся, запустите всё снова: готовые камеры будут пропущены.
5. Когда появится надпись **ГОТОВО**, скачайте с Диска `mct/perception_1280.zip` и положите рядом с `trial_results.zip`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv
!ls -la /content/drive/MyDrive/mct/mct_colab.zip


In [ ]:
%%bash
set -e
mkdir -p /content/mct
unzip -oq /content/drive/MyDrive/mct/mct_colab.zip -d /content/mct
pip install -q boxmot==26.0.0 ultralytics==8.4.203 imageio-ffmpeg lap pycocotools pyarrow
ls /content/mct /content/mct/data


In [ ]:
import torch, boxmot, ultralytics
print('cuda', torch.cuda.is_available(), 'boxmot', boxmot.__version__, 'ultralytics', ultralytics.__version__)
assert torch.cuda.is_available(), 'GPU не подключён: смените среду выполнения'


## Детекция

Камеры считаются одновременно, если процессоров хватает. Строки ниже обновляются раз в минуту: сколько кадров из примерно 12000 готово на каждой камере.


In [ ]:
import os, shutil, subprocess, time
from pathlib import Path

MODEL, IMGSZ, SCALE = 'yolo11m-seg.pt', 1280, 1
ROOT = Path('/content/mct')
OUT = Path('/content/drive/MyDrive/mct/perception_1280')
OUT.mkdir(parents=True, exist_ok=True)
ENV = {**os.environ, 'PYTHONPATH': 'src'}
PARALLEL = max(1, min(5, (os.cpu_count() or 2) // 2))
print('камер одновременно:', PARALLEL)

# Веса скачиваются один раз, до одновременного запуска
subprocess.run(['python', '-c', f"from ultralytics import YOLO; YOLO('{MODEL}'); from mct.embedder import create_embedder; create_embedder('0')"], cwd=ROOT, env=ENV, check=True)

def last_line(path):
    lines = [line for line in path.read_text(errors='ignore').splitlines() if 'кадров' in line or 'Error' in line or 'Traceback' in line]
    return lines[-1][-60:] if lines else 'запуск'

waiting = sorted(ROOT.glob('data/*.mkv'))
running = {}
failed = []
while waiting or running:
    while waiting and len(running) < PARALLEL:
        video = waiting.pop(0)
        folder = ROOT / 'one' / video.stem
        folder.mkdir(parents=True, exist_ok=True)
        if not (folder / video.name).exists():
            (folder / video.name).symlink_to(video)
        log = Path(f'/content/{video.stem}.log')
        running[video.stem] = (subprocess.Popen(
            ['python', 'scripts/run_perception.py', str(folder), '--out', str(OUT), '--device', '0', '--model', MODEL, '--imgsz', str(IMGSZ), '--detect-scale', str(SCALE)],
            cwd=ROOT, env=ENV, stdout=log.open('a'), stderr=subprocess.STDOUT), log)
    time.sleep(60)
    print(time.strftime('%H:%M'), {name: last_line(log) for name, (_, log) in running.items()}, flush=True)
    for name, (job, log) in list(running.items()):
        if job.poll() is not None:
            print(name, 'закончила, код', job.returncode, flush=True)
            if job.returncode:
                failed.append(name)
                print(log.read_text(errors='ignore')[-1500:], flush=True)
            del running[name]
print('ошибки:', failed or 'нет')
print(len(list(OUT.glob('*_summary.json'))), 'камер готово из 5')


## Снимки внешности чаще раза в секунду


In [ ]:
subprocess.run(['python', 'scripts/reembed.py', 'data', '--perception', str(OUT), '--every', '3', '--kinds', 'box', '--out', str(OUT / 'dense'), '--device', '0'], cwd=ROOT, env=ENV, check=True)
archive = shutil.make_archive('/content/drive/MyDrive/mct/perception_1280', 'zip', OUT)
print('ГОТОВО:', archive, round(os.path.getsize(archive) / 1e6), 'МБ')
